# C2 Repeat Run: Baseline vs UPR-CRE v0.2 on RegDB, T4x2

This notebook is a **runner only**. It does not edit source code.

Goal:
- Repeat the main Step 7 result under the same phase-2 15-epoch setting.
- Run baseline and UPR-CRE v0.2 in parallel on Kaggle T4x2.
- Collect Rank-1 / mAP / mINP and relation diagnostics.
- Archive results and optionally push small logs/CSV/manifest to GitHub using the old `git push` method.

Main comparison:
- GPU 0: baseline `phase2=15`, no UPR.
- GPU 1: UPR-CRE v0.2 `beta=0.1`, `gamma=0.0`, `warmup=2`, `filter_start_ratio=0.55`, `filter_end_ratio=1.0`, `phase2=15`.


## Block 0 — Configuration

Change only this block if needed. The default RegDB source matches your Kaggle dataset mount.

In [1]:
from pathlib import Path
from datetime import datetime

CFG = {
    "GITHUB_OWNER": "TranTruongMMCII",
    "REPO_NAME": "UIT.CS2309",
    "BRANCH": "feature/upr-cre",
    "WORK_DIR": "/kaggle/working",
    "DATA_ROOT": "/kaggle/working/VIREID_Dataset",
    "REGDB_SOURCE": "/kaggle/input/datasets/xqq027/reg-db/RegDB",
    "PHASE1_CKPT_PATH": "/kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/phase1_model_5.pth",
    "SEED": 1,
    "TRIAL": 1,
    "STAGE2_EPOCH": 15,
    "MILESTONES": "8 12",
    "BATCH_PIDNUM": 5,
    "PID_NUMSAMPLE": 4,
    "TEST_BATCH": 64,
    "NUM_WORKERS": 2,
    "LR": 0.00045,
    "UPR_BETA": 0.1,
    "UPR_GAMMA": 0.0,
    "UPR_MARGIN_WEIGHT": 1.0,
    "UPR_WARMUP_EPOCH": 2,
    "UPR_FILTER_START_EPOCH": 2,
    "UPR_FILTER_END_EPOCH": 10,
    "UPR_FILTER_START_RATIO": 0.55,
    "UPR_FILTER_END_RATIO": 1.0,
    "UPR_FILTER_MIN_PAIRS": 40,
    "REF_STEP6_BASELINE_R1": 0.5816,
    "REF_STEP6_BASELINE_MAP": 0.5467,
    "REF_STEP6_BASELINE_MINP": 0.4269,
    "REF_STEP7_UPR_R1": 0.6670,
    "REF_STEP7_UPR_MAP": 0.6234,
    "REF_STEP7_UPR_MINP": 0.4897,
    "PUSH_SMALL_BACKUP_TO_GIT": True,
    "GITHUB_TOKEN_SECRET": "GITHUB_TOKEN",
    "GIT_USER_NAME": "Kaggle Bot",
    "GIT_USER_EMAIL": "kaggle-bot@example.com",
}

RUN_SUFFIX = datetime.utcnow().strftime("c2repeat_%Y%m%d_%H%M%S")
repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
wsl_dir = repo_dir / "WSL_ReID"

print("RUN_SUFFIX =", RUN_SUFFIX)
print("repo_dir =", repo_dir)
print("wsl_dir =", wsl_dir)
print("checkpoint =", CFG["PHASE1_CKPT_PATH"])
print("checkpoint exists:", Path(CFG["PHASE1_CKPT_PATH"]).exists())


RUN_SUFFIX = c2repeat_20260625_142152
repo_dir = /kaggle/working/UIT.CS2309
wsl_dir = /kaggle/working/UIT.CS2309/WSL_ReID
checkpoint = /kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/phase1_model_5.pth
checkpoint exists: True


/tmp/ipykernel_58/276274761.py:42: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_SUFFIX = datetime.utcnow().strftime("c2repeat_%Y%m%d_%H%M%S")


## Block 1 — Pull repo and checkout branch

This block resets the Kaggle copy to `origin/feature/upr-cre`. It does not change code.

In [2]:
import subprocess
from pathlib import Path

repo_url = f"https://github.com/{CFG['GITHUB_OWNER']}/{CFG['REPO_NAME']}.git"
repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]

if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", CFG["BRANCH"], repo_url, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", CFG["BRANCH"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "checkout", CFG["BRANCH"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{CFG['BRANCH']}"], cwd=repo_dir, check=True)

subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir, check=True)
subprocess.run(["git", "status", "--short"], cwd=repo_dir, check=True)


Cloning into '/kaggle/working/UIT.CS2309'...


6b875c6


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

## Block 2 — Install Kaggle requirements and prepare RegDB

This uses the scripts already committed in your repo. It also checks that two GPUs are visible.

In [3]:
%%bash
set -e
cd /kaggle/working/UIT.CS2309/WSL_ReID

pip install -q -r requirements-kaggle.txt
python scripts/apply_kaggle_compat_patches.py
python scripts/prepare_regdb_kaggle.py \
  --data-root /kaggle/working/VIREID_Dataset \
  --regdb-source /kaggle/input/datasets/xqq027/reg-db/RegDB
python scripts/check_kaggle_env.py \
  --data-root /kaggle/working/VIREID_Dataset


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
No compatibility patches were needed.
RegDB source: /kaggle/input/datasets/xqq027/reg-db/RegDB
RegDB prepared at: /kaggle/working/VIREID_Dataset/RegDB
Use training argument: --data-path /kaggle/working/VIREID_Dataset
=== PyTorch / GPU ===
torch: 2.10.0+cu128
cuda available: True
gpu count: 2
gpu 0: Tesla T4
gpu 1: Tesla T4

=== RegDB layout ===
/kaggle/working/VIREID_Dataset/RegDB/idx/train_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/train_thermal_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_thermal_1.txt OK

 train_visible_1.txt
first line: Visible/285/male_back_v_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset/RegDB/Visible/285/male_back_v_05528_285.bmp
exists: True
image size: (64, 128) mode: RGB label: 0

 train_thermal_1.txt
first line: Thermal/285/male_back_t_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset

## Block 3 — Static checks

This checks the code/scripts needed for Step 5. It does not check markdown docs.

In [4]:
from pathlib import Path
import subprocess

wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"
required_files = ["main.py", "wsl.py", "task/train.py", "relation_metrics.py", "scripts/collect_relation_stats.py"]
missing = [p for p in required_files if not (wsl_dir / p).exists()]
if missing:
    raise FileNotFoundError("Missing files: " + ", ".join(missing))
checks = {
    "main.py": ["--upr-cre", "--upr-filter", "--save-relation-stats"],
    "wsl.py": ["_apply_upr_cre_score", "_apply_upr_pair_filter", "_upr_filter_keep_ratio"],
    "scripts/collect_relation_stats.py": ["--csv-output"],
}
for rel, markers in checks.items():
    text = (wsl_dir / rel).read_text()
    for marker in markers:
        if marker not in text:
            raise RuntimeError(f"Missing marker {marker!r} in {rel}")
subprocess.run(["python", "-m", "py_compile", "main.py", "wsl.py", "relation_metrics.py", "scripts/collect_relation_stats.py"], cwd=wsl_dir, check=True)
print("Static checks OK.")


Static checks OK.


## Block 4 — Launch C2 repeat jobs in background on T4x2


In [5]:
%%bash
set -e
cd /kaggle/working/UIT.CS2309/WSL_ReID
mkdir -p /kaggle/working/run_logs /kaggle/working/pids

PHASE1_CKPT="/kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/phase1_model_5.pth"
if [ ! -f "$PHASE1_CKPT" ]; then
  echo "ERROR: missing checkpoint $PHASE1_CKPT"
  exit 1
fi

COMMIT=$(git rev-parse --short HEAD)
RUN_SUFFIX="c2repeat_$(date -u +%Y%m%d_%H%M%S)_${COMMIT}"
echo "$RUN_SUFFIX" > /kaggle/working/c2_repeat_run_suffix.txt

echo "RUN_SUFFIX=$RUN_SUFFIX"
echo "COMMIT=$COMMIT"
echo "PHASE1_CKPT=$PHASE1_CKPT"

CUDA_VISIBLE_DEVICES=0 nohup python -u main.py \
  --dataset regdb \
  --data-path /kaggle/working/VIREID_Dataset \
  --debug wsl \
  --save-path "baseline_p2s15_repeat_${RUN_SUFFIX}" \
  --arch resnet \
  --trial 1 \
  --seed 1 \
  --model-path "$PHASE1_CKPT" \
  --stage1-epoch 0 \
  --stage2-epoch 15 \
  --lr 0.00045 \
  --batch-pidnum 5 \
  --pid-numsample 4 \
  --test-batch 64 \
  --num-workers 2 \
  --device 0 \
  --milestones 8 12 \
  --save-relation-stats \
  --relation-stats-dir "../saved_regdb_resnet/baseline_p2s15_repeat_${RUN_SUFFIX}_1/relation_stats" \
  > "/kaggle/working/run_logs/baseline_p2s15_repeat_${RUN_SUFFIX}.log" 2>&1 &

echo $! > /kaggle/working/pids/c2_baseline.pid

CUDA_VISIBLE_DEVICES=1 nohup python -u main.py \
  --dataset regdb \
  --data-path /kaggle/working/VIREID_Dataset \
  --debug wsl \
  --save-path "upr_v02_filter055_p2s15_repeat_${RUN_SUFFIX}" \
  --arch resnet \
  --trial 1 \
  --seed 1 \
  --model-path "$PHASE1_CKPT" \
  --stage1-epoch 0 \
  --stage2-epoch 15 \
  --lr 0.00045 \
  --batch-pidnum 5 \
  --pid-numsample 4 \
  --test-batch 64 \
  --num-workers 2 \
  --device 0 \
  --milestones 8 12 \
  --upr-cre \
  --upr-beta 0.1 \
  --upr-gamma 0.0 \
  --upr-margin-weight 1.0 \
  --upr-warmup-epoch 2 \
  --upr-filter \
  --upr-filter-start-epoch 2 \
  --upr-filter-end-epoch 10 \
  --upr-filter-start-ratio 0.55 \
  --upr-filter-end-ratio 1.0 \
  --upr-filter-min-pairs 40 \
  --save-relation-stats \
  --relation-stats-dir "../saved_regdb_resnet/upr_v02_filter055_p2s15_repeat_${RUN_SUFFIX}_1/relation_stats" \
  > "/kaggle/working/run_logs/upr_v02_filter055_p2s15_repeat_${RUN_SUFFIX}.log" 2>&1 &

echo $! > /kaggle/working/pids/c2_upr.pid

echo "Started background jobs:"
echo "baseline PID: $(cat /kaggle/working/pids/c2_baseline.pid)"
echo "UPR PID:      $(cat /kaggle/working/pids/c2_upr.pid)"
echo "Logs:"
echo "/kaggle/working/run_logs/baseline_p2s15_repeat_${RUN_SUFFIX}.log"
echo "/kaggle/working/run_logs/upr_v02_filter055_p2s15_repeat_${RUN_SUFFIX}.log"


RUN_SUFFIX=c2repeat_20260625_142204_6b875c6
COMMIT=6b875c6
PHASE1_CKPT=/kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/phase1_model_5.pth
Started background jobs:
baseline PID: 133
UPR PID:      134
Logs:
/kaggle/working/run_logs/baseline_p2s15_repeat_c2repeat_20260625_142204_6b875c6.log
/kaggle/working/run_logs/upr_v02_filter055_p2s15_repeat_c2repeat_20260625_142204_6b875c6.log


## Block 5 — Monitor jobs


In [6]:
%%bash
RUN_SUFFIX=$(cat /kaggle/working/c2_repeat_run_suffix.txt)
echo "RUN_SUFFIX=$RUN_SUFFIX"
echo "===== GPU ====="
nvidia-smi

echo "===== PIDs ====="
cat /kaggle/working/pids/c2_*.pid 2>/dev/null || true

BASE_LOG="/kaggle/working/run_logs/baseline_p2s15_repeat_${RUN_SUFFIX}.log"
UPR_LOG="/kaggle/working/run_logs/upr_v02_filter055_p2s15_repeat_${RUN_SUFFIX}.log"

echo "===== baseline log ====="
tail -n 50 -- "$BASE_LOG" || true

echo "===== UPR log ====="
tail -n 50 -- "$UPR_LOG" || true


RUN_SUFFIX=c2repeat_20260625_142204_6b875c6
===== GPU =====
Thu Jun 25 14:22:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |            

## Block 6 — Wait until jobs finish


In [7]:
%%bash
PIDS="/kaggle/working/pids/c2_baseline.pid /kaggle/working/pids/c2_upr.pid"
echo "Waiting for PIDs:"
cat $PIDS
while true; do
  RUNNING=0``
  for PID_FILE in $PIDS; do
    PID=$(cat "$PID_FILE")
    if kill -0 "$PID" 2>/dev/null; then
      echo "$(date -u '+%H:%M:%S') still running: PID=$PID"
      RUNNING=1
    else
      echo "$(date -u '+%H:%M:%S') finished: PID=$PID"
    fi
  done
  nvidia-smi --query-compute-apps=pid,used_memory --format=csv,noheader || true
  if [ "$RUNNING" -eq 0 ]; then
    echo "All jobs finished."
    break
  fi
  sleep 60
done


Waiting for PIDs:
133
134
14:22:05 still running: PID=133
14:22:05 still running: PID=134
14:23:05 still running: PID=133
14:23:05 still running: PID=134
133, 1232 MiB
134, 1232 MiB
14:24:05 still running: PID=133
14:24:05 still running: PID=134
133, 8562 MiB
134, 8562 MiB
14:25:05 still running: PID=133
14:25:05 still running: PID=134
133, 8564 MiB
134, 8564 MiB
14:26:05 still running: PID=133
14:26:05 still running: PID=134
133, 8564 MiB
134, 8564 MiB
14:27:05 still running: PID=133
14:27:05 still running: PID=134
133, 8564 MiB
134, 8564 MiB
14:28:05 still running: PID=133
14:28:05 still running: PID=134
133, 8564 MiB
134, 8564 MiB
14:29:05 still running: PID=133
14:29:05 still running: PID=134
133, 8564 MiB
134, 8564 MiB
14:30:05 still running: PID=133
14:30:05 still running: PID=134
133, 8564 MiB
134, 8564 MiB
14:31:05 still running: PID=133
14:31:05 still running: PID=134
133, 8564 MiB
134, 8564 MiB
14:32:05 still running: PID=133
14:32:05 still running: PID=134
133, 8564 MiB
134,

## Block 7 — Collect relation stats and build summary CSV


In [8]:
%%bash
set -e
cd /kaggle/working/UIT.CS2309/WSL_ReID
RUN_SUFFIX=$(cat /kaggle/working/c2_repeat_run_suffix.txt)
echo "RUN_SUFFIX=$RUN_SUFFIX"
for RUN in \
  "baseline_p2s15_repeat_${RUN_SUFFIX}_1" \
  "upr_v02_filter055_p2s15_repeat_${RUN_SUFFIX}_1"
do
  STATS_DIR="../saved_regdb_resnet/${RUN}/relation_stats"
  echo "=============================="
  echo "Collecting $RUN"
  python scripts/collect_relation_stats.py \
    --stats-dir "$STATS_DIR" \
    --csv-output "$STATS_DIR/relation_stats_summary.csv"
  echo "---- metrics ----"
  grep -E "R1:|Best_R1|Epoch:" "../saved_regdb_resnet/${RUN}/log/log.txt" | tail -60
  echo "---- relation stats tail ----"
  tail -5 "$STATS_DIR/relation_stats_summary.csv"
done


RUN_SUFFIX=c2repeat_20260625_142204_6b875c6
../saved_regdb_resnet/baseline_p2s15_repeat_c2repeat_20260625_142204_6b875c6_1/relation_stats/relation_stats_summary.csv
---- metrics ----
Epoch: 0;Time: 2026-06-25 14:32:26;Setting: ../saved_regdb_resnet/baseline_p2s15_repeat_c2repeat_20260625_142204_6b875c6_1
R1:0.2694;   R10:0.4461;  R20:0.5311;  mAP:0.2799;  mINP:0.2103
                   Best_R1: 0.2694;    Best mAP: 0.2799
Epoch: 1;Time: 2026-06-25 14:41:55;Setting: ../saved_regdb_resnet/baseline_p2s15_repeat_c2repeat_20260625_142204_6b875c6_1
R1:0.3374;   R10:0.4903;  R20:0.5932;  mAP:0.3359;  mINP:0.2534
                   Best_R1: 0.3374;    Best mAP: 0.3359
Epoch: 2;Time: 2026-06-25 14:51:24;Setting: ../saved_regdb_resnet/baseline_p2s15_repeat_c2repeat_20260625_142204_6b875c6_1
R1:0.3728;   R10:0.5607;  R20:0.6335;  mAP:0.3812;  mINP:0.2935
                   Best_R1: 0.3728;    Best mAP: 0.3812
Epoch: 3;Time: 2026-06-25 15:00:55;Setting: ../saved_regdb_resnet/baseline_p2s15_repeat_

In [9]:
%%bash
cd /kaggle/working/UIT.CS2309/WSL_ReID
python - <<'PY'
import csv, re
from pathlib import Path
run_suffix = Path('/kaggle/working/c2_repeat_run_suffix.txt').read_text().strip()
base = Path('../saved_regdb_resnet')
runs = [
    {'tag':'reference_step6_baseline_p2s15','save_path':'reference','enable_upr':False,'beta':0.0,'gamma':0.0,'filter_start_ratio':'','filter_end_ratio':'','best_r1_by_map':'0.5816','best_map':'0.5467','best_minp':'0.4269','last_common_accuracy':'','last_mutual_match_ratio':''},
    {'tag':'reference_step7_upr_v02_filter055_p2s15','save_path':'reference','enable_upr':True,'beta':0.1,'gamma':0.0,'filter_start_ratio':0.55,'filter_end_ratio':1.0,'best_r1_by_map':'0.6670','best_map':'0.6234','best_minp':'0.4897','last_common_accuracy':'0.8070','last_mutual_match_ratio':'0.9607'},
    {'tag':'repeat_baseline_p2s15','save_path':f'baseline_p2s15_repeat_{run_suffix}','enable_upr':False,'beta':0.0,'gamma':0.0,'filter_start_ratio':'','filter_end_ratio':''},
    {'tag':'repeat_upr_v02_filter055_p2s15','save_path':f'upr_v02_filter055_p2s15_repeat_{run_suffix}','enable_upr':True,'beta':0.1,'gamma':0.0,'filter_start_ratio':0.55,'filter_end_ratio':1.0},
]
def parse_best_metrics(log_text):
    matches = re.findall(r"R1:([0-9.]+);\s+R10:([0-9.]+);\s+R20:([0-9.]+);\s+mAP:([0-9.]+);\s+mINP:([0-9.]+)", log_text)
    if not matches: return '', '', ''
    best = max(matches, key=lambda x: float(x[3]))
    return best[0], best[3], best[4]
rows=[]
for item in runs:
    row=dict(item)
    if item['save_path']!='reference':
        run_dir=base/f"{item['save_path']}_1"
        log_path=run_dir/'log'/'log.txt'
        stats_path=run_dir/'relation_stats'/'relation_stats_summary.csv'
        log_text=log_path.read_text() if log_path.exists() else ''
        r1,map_,minp=parse_best_metrics(log_text)
        row['best_r1_by_map']=r1; row['best_map']=map_; row['best_minp']=minp
        last_stats={}
        if stats_path.exists():
            with stats_path.open() as f:
                reader=list(csv.DictReader(f))
                if reader: last_stats=reader[-1]
        row['last_common_accuracy']=last_stats.get('common_accuracy','')
        row['last_mutual_match_ratio']=last_stats.get('mutual_match_ratio','')
        row['last_num_common']=last_stats.get('num_common','')
        row['last_num_specific']=last_stats.get('num_specific','')
        row['last_num_remain']=last_stats.get('num_remain','')
    else:
        row.setdefault('last_num_common',''); row.setdefault('last_num_specific',''); row.setdefault('last_num_remain','')
    rows.append(row)
fields=['tag','save_path','enable_upr','beta','gamma','filter_start_ratio','filter_end_ratio','best_r1_by_map','best_map','best_minp','last_common_accuracy','last_mutual_match_ratio','last_num_common','last_num_specific','last_num_remain']
out=Path('/kaggle/working/c2_repeat_baseline_vs_upr_summary.csv')
with out.open('w', newline='') as f:
    writer=csv.DictWriter(f, fieldnames=fields); writer.writeheader(); writer.writerows(rows)
print(out.read_text())
PY


tag,save_path,enable_upr,beta,gamma,filter_start_ratio,filter_end_ratio,best_r1_by_map,best_map,best_minp,last_common_accuracy,last_mutual_match_ratio,last_num_common,last_num_specific,last_num_remain
reference_step6_baseline_p2s15,reference,False,0.0,0.0,,,0.5816,0.5467,0.4269,,,,,
reference_step7_upr_v02_filter055_p2s15,reference,True,0.1,0.0,0.55,1.0,0.6670,0.6234,0.4897,0.8070,0.9607,,,
repeat_baseline_p2s15,baseline_p2s15_repeat_c2repeat_20260625_142204_6b875c6,False,0.0,0.0,,,0.5816,0.5467,0.4269,0.6432748538011696,0.9447513812154696,171,4,17
repeat_upr_v02_filter055_p2s15,upr_v02_filter055_p2s15_repeat_c2repeat_20260625_142204_6b875c6,True,0.1,0.0,0.55,1.0,0.6670,0.6234,0.4897,0.8070175438596491,0.9606741573033708,171,2,14



## Block 8 — Archive artifacts


In [10]:
%%bash
set -e
cd /kaggle/working/UIT.CS2309/WSL_ReID
RUN_SUFFIX=$(cat /kaggle/working/c2_repeat_run_suffix.txt)
ART_DIR="/kaggle/working/artifacts_c2_repeat_${RUN_SUFFIX}"
rm -rf "$ART_DIR"
mkdir -p "$ART_DIR/logs" "$ART_DIR/relation_stats"
cp /kaggle/working/c2_repeat_baseline_vs_upr_summary.csv "$ART_DIR/" || true
cp /kaggle/working/c2_repeat_run_suffix.txt "$ART_DIR/" || true
git rev-parse --short HEAD > "$ART_DIR/commit.txt"
for RUN in \
  "baseline_p2s15_repeat_${RUN_SUFFIX}_1" \
  "upr_v02_filter055_p2s15_repeat_${RUN_SUFFIX}_1"
do
  cp "../saved_regdb_resnet/${RUN}/log/log.txt" "$ART_DIR/logs/${RUN}_log.txt" || true
  cp "../saved_regdb_resnet/${RUN}/relation_stats/relation_stats_summary.csv" "$ART_DIR/relation_stats/${RUN}_relation_stats_summary.csv" || true
done
TAR_PATH="/kaggle/working/artifacts_c2_repeat_${RUN_SUFFIX}.tar.gz"
tar -czf "$TAR_PATH" -C /kaggle/working "$(basename "$ART_DIR")"
ls -lh "$TAR_PATH"


-rw-r--r-- 1 root root 8.3K Jun 25 17:02 /kaggle/working/artifacts_c2_repeat_c2repeat_20260625_142204_6b875c6.tar.gz


## Block 9 — Optional: push small backup to GitHub using old git push method


In [11]:
from pathlib import Path
import os, json, shutil, subprocess, textwrap
if not CFG.get('PUSH_SMALL_BACKUP_TO_GIT', False):
    print('Git backup disabled.')
else:
    from kaggle_secrets import UserSecretsClient
    WORK_DIR=Path(CFG['WORK_DIR']); REPO_DIR=WORK_DIR/CFG['REPO_NAME']
    RUN_SUFFIX=Path('/kaggle/working/c2_repeat_run_suffix.txt').read_text().strip()
    ART_DIR=Path(f'/kaggle/working/artifacts_c2_repeat_{RUN_SUFFIX}')
    TAR_PATH=Path(f'/kaggle/working/artifacts_c2_repeat_{RUN_SUFFIX}.tar.gz')
    assert ART_DIR.exists(), ART_DIR
    backup_dir=REPO_DIR/'experiments'/'kaggle_runs'/RUN_SUFFIX
    if backup_dir.exists(): shutil.rmtree(backup_dir)
    backup_dir.mkdir(parents=True, exist_ok=True)
    for sub in ['logs','relation_stats']:
        src=ART_DIR/sub
        if src.exists(): shutil.copytree(src, backup_dir/sub, dirs_exist_ok=True)
    for pattern in ['*.csv','*.txt','*.json']:
        for src in ART_DIR.glob(pattern): shutil.copy2(src, backup_dir/src.name)
    manifest={'run_id':RUN_SUFFIX,'repo':f"{CFG['GITHUB_OWNER']}/{CFG['REPO_NAME']}",'branch':CFG['BRANCH'],'backup_dir_in_repo':str(backup_dir.relative_to(REPO_DIR)),'artifact_tar':str(TAR_PATH) if TAR_PATH.exists() else None,'large_files_committed':False,'note':'Large files are not committed. Download artifact tar or upload it as Kaggle Dataset/Input if needed.'}
    (backup_dir/'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
    (backup_dir/'README.md').write_text(f"# C2 Repeat Run Backup: {RUN_SUFFIX}\n\nThis folder was pushed from Kaggle using normal git push.\n")
    token=UserSecretsClient().get_secret(CFG.get('GITHUB_TOKEN_SECRET','GITHUB_TOKEN')).strip()
    askpass=WORK_DIR/'git_askpass_push.sh'
    askpass_text = '#!/bin/sh\ncase "$1" in\n  *Username*) echo "' + CFG['GITHUB_OWNER'] + '" ;;\n  *Password*) echo "' + token + '" ;;\n  *) echo "" ;;\nesac\n'
    askpass.write_text(askpass_text); askpass.chmod(0o700)
    env=os.environ.copy(); env['GIT_ASKPASS']=str(askpass); env['GIT_TERMINAL_PROMPT']='0'
    def run(cmd):
        print('+',' '.join(cmd)); return subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
    try:
        run(['git','config','user.name',CFG.get('GIT_USER_NAME','Kaggle Bot')])
        run(['git','config','user.email',CFG.get('GIT_USER_EMAIL','kaggle-bot@example.com')])
        run(['git','remote','set-url','origin',f"https://github.com/{CFG['GITHUB_OWNER']}/{CFG['REPO_NAME']}.git"])
        run(['git','fetch','origin',CFG['BRANCH']]); run(['git','checkout',CFG['BRANCH']]); run(['git','pull','--rebase','origin',CFG['BRANCH']])
        rel_backup=str(backup_dir.relative_to(REPO_DIR)); run(['git','add','-f',rel_backup])
        diff=subprocess.run(['git','diff','--cached','--quiet'], cwd=REPO_DIR, env=env)
        if diff.returncode==0: print('No staged changes to commit.')
        else:
            run(['git','commit','-m',f'exp: add C2 repeat backup {RUN_SUFFIX}']); run(['git','push','origin',CFG['BRANCH']])
        print(f"https://github.com/{CFG['GITHUB_OWNER']}/{CFG['REPO_NAME']}/tree/{CFG['BRANCH']}/experiments/kaggle_runs/{RUN_SUFFIX}")
    finally:
        try: askpass.unlink()
        except FileNotFoundError: pass


+ git config user.name Kaggle Bot
+ git config user.email kaggle-bot@example.com
+ git remote set-url origin https://github.com/TranTruongMMCII/UIT.CS2309.git
+ git fetch origin feature/upr-cre


From https://github.com/TranTruongMMCII/UIT.CS2309
 * branch            feature/upr-cre -> FETCH_HEAD
Already on 'feature/upr-cre'


+ git checkout feature/upr-cre
Your branch is up to date with 'origin/feature/upr-cre'.
+ git pull --rebase origin feature/upr-cre


From https://github.com/TranTruongMMCII/UIT.CS2309
 * branch            feature/upr-cre -> FETCH_HEAD


Already up to date.
+ git add -f experiments/kaggle_runs/c2repeat_20260625_142204_6b875c6
+ git commit -m exp: add C2 repeat backup c2repeat_20260625_142204_6b875c6
[feature/upr-cre 04b75af] exp: add C2 repeat backup c2repeat_20260625_142204_6b875c6
 9 files changed, 415 insertions(+)
 create mode 100644 experiments/kaggle_runs/c2repeat_20260625_142204_6b875c6/README.md
 create mode 100644 experiments/kaggle_runs/c2repeat_20260625_142204_6b875c6/c2_repeat_baseline_vs_upr_summary.csv
 create mode 100644 experiments/kaggle_runs/c2repeat_20260625_142204_6b875c6/c2_repeat_run_suffix.txt
 create mode 100644 experiments/kaggle_runs/c2repeat_20260625_142204_6b875c6/commit.txt
 create mode 100644 experiments/kaggle_runs/c2repeat_20260625_142204_6b875c6/logs/baseline_p2s15_repeat_c2repeat_20260625_142204_6b875c6_1_log.txt
 create mode 100644 experiments/kaggle_runs/c2repeat_20260625_142204_6b875c6/logs/upr_v02_filter055_p2s15_repeat_c2repeat_20260625_142204_6b875c6_1_log.txt
 create mode 100644

To https://github.com/TranTruongMMCII/UIT.CS2309.git
   6b875c6..04b75af  feature/upr-cre -> feature/upr-cre
